# Patent phrase matching — controlled experiments

Source: official course starter/project brief, bk-anupam/generative_ai_shared master, accessed 5 November 2026. This revision preserves the official baseline settings and adds a pretrained bi-encoder, two one-factor context experiments, error analysis and evidence export.

**Status (12 November 2026):** Tasks 1-5 are complete. The experiments and error analysis below were executed locally. The final configuration was executed offline and submitted on Kaggle with user approval. The final report and appended submission record contain actual scores. Only official competition training labels were used.

Set PATENT_DATA_DIR to the extracted competition folder locally, or attach the competition in Kaggle. The full experiment notebook runs three training jobs. Use the validation-selected setting in the separate final_submission.ipynb for a single offline final run.

## 1. Setup

In [1]:
import glob
import os

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
transformers.set_seed(SEED)   # seeds Python, NumPy, and PyTorch; GPU results can still vary slightly between runs
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)
import json, time, gc
from pathlib import Path
from scipy.stats import pearsonr


/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.10.0+cu128 | transformers 4.57.1 | device: cuda


## 2. Configuration

These are the settings you will change in your experiments. Change **one** setting at a time so you can tell what caused a difference.

In [2]:
MODEL_ID = "microsoft/deberta-v3-small"
MAX_LEN, EPOCHS, LR, BATCH_SIZE = 64, 3, 2e-5, 32
VAL_FRACTION = 0.2
OUT_DIR = os.environ.get("PATENT_OUTPUT_DIR") or ("/kaggle/working" if Path("/kaggle/working").is_dir() else "patent_outputs")
Path(OUT_DIR).mkdir(parents=True,exist_ok=True)
def find_one(pattern):
    hits=sorted(glob.glob(f"/kaggle/input/**/{pattern}",recursive=True))
    if len(hits)>1:
        raise ValueError(f"Ambiguous attached input for {pattern}; set an explicit path.")
    return os.path.dirname(hits[0]) if hits else None
DATA_DIR=os.environ.get("PATENT_DATA_DIR") or find_one("us-patent-phrase-to-phrase-matching/train.csv")
assert DATA_DIR, "Attach the official competition data or set PATENT_DATA_DIR. Do not use the later full rated dataset."
MODEL_PATH=os.environ.get("PATENT_MODEL_PATH") or find_one(f"{MODEL_ID.split('/')[-1]}/config.json") or MODEL_ID
print("data:", DATA_DIR, "| model:", MODEL_PATH)


data: /home/vansh/Documents/Codex/2026-11-05/https-github-com-bk-anupam-generative/work/data/us-patent-phrase-to-phrase-matching | model: /home/vansh/Documents/Codex/2026-11-05/https-github-com-bk-anupam-generative/work/cache/huggingface/hub/models--microsoft--deberta-v3-small/snapshots/a36c739020e01763fe789b4b85e2df55d6180012


## 3. Data

Each row asks how similar a **target** phrase is to an **anchor** phrase, judged within a patent **context**. The context is a CPC (Cooperative Patent Classification) code such as `G02`, whose first letter is the section. Scores are 0, 0.25, 0.5, 0.75, or 1. The visible `test.csv` has only a few rows; when you submit, Kaggle reruns the notebook with the hidden test set in its place.

In [3]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "| test (visible):", test.shape)
print(train.score.value_counts().sort_index())
train.head()
assert len(train)==36473, "Use only the official 36,473-row competition training set."
assert {"id","anchor","target","context","score"} <= set(train.columns)
assert {"id","anchor","target","context"} <= set(test.columns)
# The official 36-row visible test file is a format sample whose IDs occur in training.
# Never score on it or look up hidden-test labels; validation holds out whole anchors.
print("Current test IDs also present in training:", len(set(train.id) & set(test.id)))
assert train.score.isin([0,.25,.5,.75,1]).all()
raw_train,raw_test=train.copy(),test.copy()

assert "score" not in test.columns, "Do not use released hidden-test labels."
assert train.id.is_unique and test.id.is_unique
assert train[["id","anchor","target","context","score"]].notna().all().all()
assert test[["id","anchor","target","context"]].notna().all().all()


train: (36473, 5) | test (visible): (36, 4)
score
0.00     7471
0.25    11519
0.50    12300
0.75     4029
1.00     1154
Name: count, dtype: int64
Current test IDs also present in training: 36


## 4. Model input

The model reads the anchor as the first text segment, and the target plus a short context description as the second. The tokenizer joins them as `[CLS] anchor [SEP] target; context [SEP]`, so attention can compare the two phrases token by token: this is what makes it a cross-encoder. Here the context is just the CPC section name. How you represent context is one of the experiments you can run.

In [4]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}


def add_inputs(df):
    df = df.copy()
    df["text_a"] = df["anchor"]
    df["text_b"] = df["target"] + "; " + df["context"].str[0].map(CPC_SECTIONS)
    return df


train, test = add_inputs(train), add_inputs(test)
train[["text_a", "text_b", "score"]].head()

,text_a,text_b,score
0,abatement,abatement of pollution; human necessities,0.50
1,abatement,act of abating; human necessities,0.75
2,abatement,active catalyst; human necessities,0.25
3,abatement,eliminating process; human necessities,0.50
4,abatement,forest region; human necessities,0.00


## 5. Validation split

The training set contains only about 730 distinct anchors, each paired with many targets. If the same anchor appeared in both the training and validation sets, the validation score would overestimate performance on unseen anchors. The split therefore holds out whole **anchors**.

Keep this split fixed (same `SEED` and `VAL_FRACTION`) for every experiment, so your comparisons are fair.

In [5]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")
SPLIT_IDS={"train":trn.id.tolist(),"validation":val.id.tolist()}
Path(OUT_DIR,"split_ids.json").write_text(json.dumps(SPLIT_IDS,indent=2))


train rows 28914 | val rows 7559 | val anchors 147


875393

## 6. Tokenization

Padding is left to the data collator, which pads each batch only to its longest sequence. The printed example should start with `[CLS]` and have a `[SEP]` between the two segments.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# transformers 5.0 builds this tokenizer without [CLS]/[SEP]; set the pair template explicitly
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)])


def to_dataset(df, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    ds = Dataset.from_pandas(df[cols].astype({"score": "float32"}) if with_labels else df[cols])  # MSE loss needs float32 labels
    ds = ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


ds_trn, ds_val, ds_test = to_dataset(trn), to_dataset(val), to_dataset(test, with_labels=False)
print(tokenizer.decode(ds_trn[0]["input_ids"]))

/home/vansh/Documents/Codex/2026-11-05/https-github-com-bk-anupam-generative/work/runtime/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/28914 [00:00<?, ? examples/s]

Map:  24%|██▍       | 7000/28914 [00:00<00:00, 60552.70 examples/s]

Map:  52%|█████▏    | 15000/28914 [00:00<00:00, 64673.89 examples/s]

Map:  80%|███████▉  | 23000/28914 [00:00<00:00, 66246.98 examples/s]

Map: 100%|██████████| 28914/28914 [00:00<00:00, 64129.92 examples/s]

... (2 outputs left out)

Map: 100%|██████████| 7559/7559 [00:00<00:00, 63655.84 examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

Map: 100%|██████████| 36/36 [00:00<00:00, 17323.88 examples/s]

[CLS] abatement[SEP] abatement of pollution; human necessities[SEP]


## 7. Fine-tuning

With num_labels=1 and floating labels, the Transformer uses a regression head and mean squared error. The competition metric is Pearson correlation between predictions and true scores. Pearson measures linear association: it is unchanged by adding a constant or multiplying predictions by a positive constant, so it does not assess absolute calibration. Multiplying by a negative constant reverses its sign. It is not merely a ranking metric: a nonlinear monotone transformation can preserve every rank but change Pearson (Spearman measures rank correlation).


In [7]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": float(np.corrcoef(preds.reshape(-1), labels)[0, 1])}


# Load in fp32: the deberta-v3 checkpoints are stored in fp16, and fp16=True mixed precision below needs
# fp32 weights ("Attempting to unscale FP16 gradients" otherwise). Older transformers call this torch_dtype.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=1, torch_dtype=torch.float32)
args = TrainingArguments(
    output_dir=f"{OUT_DIR}/trainer_tmp",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2 * BATCH_SIZE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",          # keeps disk use and runtime low; the final weights stay in memory
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
start=time.perf_counter()
trainer.train()
if torch.cuda.is_available():torch.cuda.synchronize()
baseline_seconds=time.perf_counter()-start
baseline_peak_gb=torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else None
print("Baseline seconds:",baseline_seconds,"peak GB:",baseline_peak_gb)

`torch_dtype` is deprecated! Use `dtype` instead!


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at /home/vansh/Documents/Codex/2026-11-05/https-github-com-bk-anupam-generative/work/cache/huggingface/hub/models--microsoft--deberta-v3-small/snapshots/a36c739020e01763fe789b4b85e2df55d6180012 and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Pearson
1,0.029500,0.027273,0.802675
2,0.022200,0.024533,0.815156
3,0.018400,0.024572,0.817791


Baseline seconds: 208.8717847529333 peak GB: 2.72960768


## 8. Validation predictions

The predictions are saved so you can do your error analysis without retraining.

In [8]:
val["pred"] = trainer.predict(ds_val).predictions.reshape(-1)
val_pearson = np.corrcoef(val["pred"], val["score"])[0, 1]
print(f"Validation Pearson: {val_pearson:.4f}")
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)
baseline_val=val[["id","anchor","target","context","score","pred"]].copy()
baseline_val.to_csv(Path(OUT_DIR,"val_baseline.csv"),index=False)
baseline_test=trainer.predict(ds_test).predictions.reshape(-1)
experiment_results=[{"name":"baseline_section_title","changed_factor":"none (official baseline)",
    "context_mode":"section_title","pearson":float(val_pearson),"seconds":baseline_seconds,
    "peak_gb":baseline_peak_gb,"trainable":sum(p.numel() for p in model.parameters() if p.requires_grad)}]
validation_predictions={"baseline_section_title":baseline_val}
test_predictions={"baseline_section_title":baseline_test}
Path(OUT_DIR,"experiment_results.json").write_text(json.dumps(experiment_results,indent=2))
Path(OUT_DIR,"baseline_training_history.json").write_text(json.dumps(trainer.state.log_history,indent=2))
np.save(Path(OUT_DIR,"test_baseline.npy"),baseline_test)
trainer.save_model(str(Path(OUT_DIR,"checkpoint_baseline_section_title")))
tokenizer.save_pretrained(str(Path(OUT_DIR,"checkpoint_baseline_section_title")))
del trainer,model
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()


Validation Pearson: 0.8178


## Tasks 1–2: split explanation and pretrained baseline

An anchor occurs with many targets; random rows can put familiar anchors into validation and inflate generalization estimates. Grouping by anchor reduces that leakage, and the identical split makes comparisons fair.

The pretrained sentence-transformers/all-MiniLM-L6-v2 bi-encoder encodes phrases separately and compares normalized embeddings by cosine similarity. The cross-encoder attends jointly over both phrases and context, then learns from patent labels. The comparison changes both pretrained architecture and training, so it does not isolate fine-tuning as the only causal factor.

Source: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2 (Apache-2.0). No training or fitting uses validation labels for this baseline.

In [9]:
from sentence_transformers import SentenceTransformer
BI_MODEL_ID="sentence-transformers/all-MiniLM-L6-v2"
BI_MODEL_PATH=os.environ.get("PATENT_BI_MODEL_PATH") or find_one("all-MiniLM-L6-v2/config.json") or BI_MODEL_ID
bi=SentenceTransformer(BI_MODEL_PATH,device=device)
start=time.perf_counter()
a=bi.encode(val.anchor.tolist(),batch_size=128,normalize_embeddings=True,show_progress_bar=True)
b=bi.encode(val.target.tolist(),batch_size=128,normalize_embeddings=True,show_progress_bar=True)
bi_pred=(a*b).sum(axis=1)
bi_seconds=time.perf_counter()-start
bi_pearson=float(pearsonr(val.score.to_numpy(),bi_pred).statistic)
print("Bi-encoder Pearson:",bi_pearson,"inference seconds:",bi_seconds)
val.assign(pred=bi_pred)[["id","anchor","target","context","score","pred"]].to_csv(Path(OUT_DIR,"val_biencoder.csv"),index=False)
Path(OUT_DIR,"biencoder_results.json").write_text(json.dumps({"model":BI_MODEL_ID,"pearson":bi_pearson,"inference_seconds":bi_seconds,"training_seconds":0},indent=2))
del bi,a,b
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Batches:  17%|█▋        | 10/60 [00:00<00:00, 99.50it/s]

Batches:  40%|████      | 24/60 [00:00<00:00, 122.50it/s]

Batches:  67%|██████▋   | 40/60 [00:00<00:00, 135.91it/s]

Batches:  95%|█████████▌| 57/60 [00:00<00:00, 146.06it/s]

Batches: 100%|██████████| 60/60 [00:00<00:00, 138.60it/s]

... (4 outputs left out)

Batches:  75%|███████▌  | 45/60 [00:00<00:00, 113.68it/s]

Batches: 100%|██████████| 60/60 [00:00<00:00, 125.97it/s]

Batches: 100%|██████████| 60/60 [00:00<00:00, 114.11it/s]

Bi-encoder Pearson: 0.5814485416179851 inference seconds: 0.9855052139610052


## Task 3: two controlled context experiments

Each experiment starts from the same fresh pretrained checkpoint with the same seed, exact split, schedule and budget. Change only context to (1) none or (2) raw CPC code. Compare each against the original section-title baseline. A single-seed difference around 0.005 is inconclusive; do not claim a general improvement.

In [10]:
def context_inputs(df,mode):
    df=df.copy()
    df["text_a"]=df.anchor
    suffix={"none":pd.Series("",index=df.index),"raw_code":"; "+df.context,
            "section_title":"; "+df.context.str[0].map(CPC_SECTIONS)}[mode]
    df["text_b"]=df.target+suffix
    return df

def run_experiment(mode):
    transformers.set_seed(SEED)
    table=context_inputs(raw_train,mode)
    exp_trn=table.iloc[train_idx].reset_index(drop=True)
    exp_val=table.iloc[val_idx].reset_index(drop=True)
    assert exp_trn.id.tolist()==SPLIT_IDS["train"] and exp_val.id.tolist()==SPLIT_IDS["validation"]
    assert set(exp_trn.anchor).isdisjoint(exp_val.anchor)
    exp_test=context_inputs(raw_test,mode)
    ds_a,ds_b,ds_c=to_dataset(exp_trn),to_dataset(exp_val),to_dataset(exp_test,False)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL_PATH,num_labels=1,torch_dtype=torch.float32)
    args=TrainingArguments(output_dir=str(Path(OUT_DIR,mode)),num_train_epochs=EPOCHS,learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,per_device_eval_batch_size=2*BATCH_SIZE,
        warmup_ratio=.1,weight_decay=.01,eval_strategy="epoch",save_strategy="no",logging_steps=100,
        fp16=torch.cuda.is_available(),report_to="none",seed=SEED)
    trainer=Trainer(model=model,args=args,train_dataset=ds_a,eval_dataset=ds_b,
        data_collator=DataCollatorWithPadding(tokenizer),compute_metrics=compute_metrics)
    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats();torch.cuda.synchronize()
    start=time.perf_counter()
    trainer.train()
    if torch.cuda.is_available():torch.cuda.synchronize()
    seconds=time.perf_counter()-start
    peak=torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else None
    exp_val["pred"]=trainer.predict(ds_b).predictions.reshape(-1)
    test_pred=trainer.predict(ds_c).predictions.reshape(-1)
    result={"name":"context_"+mode,"changed_factor":"context only","context_mode":mode,
        "pearson":float(pearsonr(exp_val.score,exp_val.pred).statistic),"seconds":seconds,
        "peak_gb":peak,"trainable":sum(p.numel() for p in model.parameters() if p.requires_grad)}
    pred=exp_val[["id","anchor","target","context","score","pred"]].copy()
    pred.to_csv(Path(OUT_DIR,"val_"+mode+".csv"),index=False)
    Path(OUT_DIR,"result_"+mode+".json").write_text(json.dumps(result,indent=2))
    Path(OUT_DIR,"history_"+mode+".json").write_text(json.dumps(trainer.state.log_history,indent=2))
    np.save(Path(OUT_DIR,"test_"+mode+".npy"),test_pred)
    trainer.save_model(str(Path(OUT_DIR,"checkpoint_context_"+mode)))
    tokenizer.save_pretrained(str(Path(OUT_DIR,"checkpoint_context_"+mode)))
    del trainer,model
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return result,pred,test_pred

for mode in ("none","raw_code"):
    result,pred,test_pred=run_experiment(mode)
    experiment_results.append(result)
    validation_predictions[result["name"]]=pred
    test_predictions[result["name"]]=test_pred
    Path(OUT_DIR,"experiment_results.json").write_text(json.dumps(experiment_results,indent=2))
results_table=pd.DataFrame(experiment_results)
results_table["delta_vs_baseline"]=results_table.pearson-results_table.iloc[0].pearson
results_table["interpretation"]=results_table.delta_vs_baseline.map(lambda d:
    "baseline" if d==0 else "small change; inconclusive" if abs(d)<=.005 else
    "higher on this split; needs repeated seeds" if d>0 else "lower on this split")
results_table.to_csv(Path(OUT_DIR,"experiment_results.csv"),index=False)
results_table


Map:   0%|          | 0/28914 [00:00<?, ? examples/s]

Map:  35%|███▍      | 10000/28914 [00:00<00:00, 86879.06 examples/s]

Map:  69%|██████▉   | 20000/28914 [00:00<00:00, 90172.08 examples/s]

Map: 100%|██████████| 28914/28914 [00:00<00:00, 89740.87 examples/s]

Map:   0%|          | 0/7559 [00:00<?, ? examples/s]

Map: 100%|██████████| 7559/7559 [00:00<00:00, 80836.76 examples/s]

... (13 outputs left out)

Map: 100%|██████████| 36/36 [00:00<00:00, 17158.52 examples/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at /home/vansh/Documents/Codex/2026-11-05/https-github-com-bk-anupam-generative/work/cache/huggingface/hub/models--microsoft--deberta-v3-small/snapshots/a36c739020e01763fe789b4b85e2df55d6180012 and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Pearson
1,0.028500,0.028101,0.802538
2,0.021700,0.024467,0.816580
3,0.019000,0.023935,0.819542


,name,changed_factor,context_mode,pearson,seconds,peak_gb,trainable,delta_vs_baseline,interpretation
0,baseline_section_title,none (official baseline),section_title,0.817791,208.871785,2.729608,141895681,0.000000,baseline
1,context_none,context only,none,0.816769,198.071460,2.655119,141895681,-0.001022,small change; inconclusive
2,context_raw_code,context only,raw_code,0.819542,202.714914,2.706164,141895681,0.001751,small change; inconclusive


## Task 4: error analysis and validation-only selection

Select by validation Pearson before viewing any leaderboard score. Correlation is undefined for constant values or fewer than two observations; report this as missing. Explain the hardest CPC sections together with sample counts, compression of extreme labels, and two or three concrete linguistic error types among the ten largest-error pairs. These interpretations must follow actual outputs.

In [11]:
import matplotlib.pyplot as plt
best=max(experiment_results,key=lambda r:r["pearson"])
best_val=validation_predictions[best["name"]].copy()
best_val["section"]=best_val.context.str[0]
def safe_pearson(frame):
    if len(frame)<2 or frame.score.nunique()<2 or frame.pred.nunique()<2:return float("nan")
    return float(pearsonr(frame.score,frame.pred).statistic)
section_table=pd.DataFrame([{"section":s,"n":len(g),"pearson":safe_pearson(g)}
    for s,g in best_val.groupby("section")]).set_index("section").reindex(list("ABCDEFGH"))
score_table=best_val.groupby("score").agg(n=("pred","size"),mean_prediction=("pred","mean"),
    sd_prediction=("pred","std")).reindex([0,.25,.5,.75,1])
largest_errors=best_val.assign(abs_error=(best_val.pred-best_val.score).abs()).sort_values("abs_error",ascending=False).head(10)
best_val.to_csv(Path(OUT_DIR,"val_predictions.csv"),index=False)
section_table.to_csv(Path(OUT_DIR,"section_analysis.csv"))
score_table.to_csv(Path(OUT_DIR,"score_analysis.csv"))
largest_errors.to_csv(Path(OUT_DIR,"largest_errors.csv"),index=False)
display(section_table,score_table,largest_errors)
fig,axes=plt.subplots(1,2,figsize=(10,3.5))
section_table.pearson.plot.bar(ax=axes[0],ylim=(-1,1),title="Pearson by CPC section")
axes[0].set_ylabel("Pearson r")
axes[1].plot(score_table.index,score_table.mean_prediction,marker="o",label="mean prediction")
axes[1].plot([0,1],[0,1],"--",label="ideal calibration")
axes[1].set(xlabel="True score",ylabel="Mean prediction",title="Score compression")
axes[1].legend();fig.tight_layout();fig.savefig(Path(OUT_DIR,"error_analysis.png"),dpi=160);plt.show()
print("Validation-selected configuration:",best)
Path(OUT_DIR,"selected_config.json").write_text(json.dumps(best,indent=2))


,n,pearson
section,,
A,943,0.777844
B,1926,0.898532
C,1177,0.766652
D,199,0.760824
E,373,0.811866
F,968,0.848998
G,1040,0.699086
H,933,0.808962


,n,mean_prediction,sd_prediction
score,,,
0.00,1522,0.109797,0.152094
0.25,2465,0.298342,0.115960
0.50,2475,0.481593,0.144328
0.75,832,0.652611,0.166991
1.00,265,1.011268,0.137999


,id,anchor,target,context,score,pred,section,abs_error
5836,6af4f2d9506545f2,punch face,face punch,B05,0.00,0.970703,B,0.970703
3794,20a45b67a785caf3,imaging axis,imaging of axis,G02,0.00,0.811523,G,0.811523
972,f297ea79eab86071,boom hydraulic cylinder,boom a hydraulic cylinder,E02,0.00,0.792480,E,0.792480
3790,c9f4b17fdba12548,imaging axis,image of axis,G02,0.00,0.779785,G,0.779785
6005,31677aa02d309ada,return structure,return to structure,D01,0.00,0.736328,D,0.736328
125,fbf3be656271239a,activating position,active region,G06,0.00,0.713379,G,0.713379
1154,12c1f4b230252489,breech mechanism,breech loading naval gun,B29,0.75,0.043243,B,0.706757
6382,d828396508aab8f0,secure to guard,security guard,A01,0.00,0.654785,A,0.654785
2944,361ad90e63ebf1e4,extracting process,process mining,C13,0.00,0.649414,C,0.649414
2776,463717d347b289d7,end pins,wood pin,G10,0.75,0.104736,G,0.645264


Validation-selected configuration: {'name': 'context_raw_code', 'changed_factor': 'context only', 'context_mode': 'raw_code', 'pearson': 0.8195418886177807, 'seconds': 202.7149142830167, 'peak_gb': 2.706163712, 'trainable': 141895681}


/tmp/ipykernel_2862291/1269679203.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  axes[1].legend();fig.tight_layout();fig.savefig(Path(OUT_DIR,"error_analysis.png"),dpi=160);plt.show()


215

## Task 5: local predictions and submission evidence

The CSV below checks format on visible test rows; it is not a hidden-test submission. The final notebook must load attached model files offline and rerun on Kaggle, after explicit user approval. Record every actual submission, public/private scores, notebook version and screenshot. Explain score differences using validation selection, anchor/section mixture and sample size.

In [12]:
submission=pd.DataFrame({"id":raw_test.id,"score":np.clip(test_predictions[best["name"]],0,1)})
assert list(submission.columns)==["id","score"] and len(submission)==len(raw_test)
assert submission.id.tolist()==raw_test.id.tolist() and np.isfinite(submission.score).all()
submission.to_csv(Path(OUT_DIR,"submission.csv"),index=False)
evidence={"seed":SEED,"validation_fraction":VAL_FRACTION,"baseline":experiment_results[0],
    "biencoder":{"model":BI_MODEL_ID,"pearson":bi_pearson,"inference_seconds":bi_seconds},
    "experiments":experiment_results,"selected":best,"submission_status":"not submitted",
    "public_score":None,"private_score":None,"all_kaggle_submissions":[]}
Path(OUT_DIR,"run_evidence.json").write_text(json.dumps(evidence,indent=2))
display(submission.head())


,id,score
0,4112d61851461f60,0.552246
1,09e418c93a776564,0.757324
2,36baf228038e314b,0.539062
3,1f37ead645e7f0c8,0.350098
4,71a5b6ad068d531f,0.000000


## Measured answers and conclusions

### Measured experiment interpretation

Removing context changes r by -0.0010; replacing section titles with raw CPC codes changes it by +0.0018. Both are smaller than the lab's 0.005 noise caution, so neither establishes a reliable improvement. Raw codes are selected by the highest observed validation score, without claiming they are generally better.

The trained baseline exceeds the pretrained bi-encoder by 0.2363 Pearson points on the same rows. This is a substantial observed system-level difference, but the comparison also changes architecture and pretraining. The three context runs together took 10.16 minutes of measured training time.

### CPC sections

Section G is hardest here (r = 0.6991, n = 1,040), followed by D (0.7608, n = 199) and C (0.7667, n = 1,177). B is strongest (0.8985, n = 1,926). D has relatively few examples, so its estimate is less stable; section rankings are descriptive rather than evidence of intrinsic difficulty.

G contributes four of the ten largest errors, spanning G02, G06 and G10. They involve phrase structure, related spatial terms and a technical component relation. A plausible explanation is that lexical resemblance is easier to learn than these distinctions from short phrases and a code. This hypothesis needs broader error review; ten extreme cases cannot establish the cause of all section-level differences.

### Score compression

There is partial compression: true 0 averages 0.110 and true 0.75 averages 0.653. Scores 0.25 and 0.5 make up 65.3% of the training data, so an MSE-trained model can favor central predictions when uncertain. But compression is not universal: true 1 averages 1.011, slightly above the valid range. The regression head is unconstrained; clipping is applied only to submission predictions.

### Three failure groups

**S - Phrase structure (rows 1-5).** Shared words overwhelm changes in word order or grammatical role. 'Punch face' names a possible part, while 'face punch' can describe an action. Likewise, an 'imaging axis' differs from imaging an axis, and a 'return structure' differs from returning to a structure. The model assigns high similarity despite zero labels.

**M - Meaning substitution (rows 6, 8, 9).** 'Activating' and 'active', 'secure' and 'security', or 'extracting' and 'mining' invite lexical associations. Those associations do not establish that the phrases have the same function in a patent. These predictions range from 0.649 to 0.713 despite zero labels.

**R - Technical relations (rows 7, 10).** The model misses the labelled similarity between a breech mechanism and a breech-loading gun, and between end pins and a wood pin. Component-to-system or role-to-material relationships may require more context than the short input provides. These are the two severe underestimates; the other eight errors are overestimates.

### Label ambiguity

**Debatable example:** 'boom hydraulic cylinder' versus 'boom a hydraulic cylinder' differs only by an added article and has an awkward construction, making its zero label surprising. Without the original patent sentences or expert rationale, I cannot resolve whether this is a meaningful distinction or annotation ambiguity. I retain the supplied label and report the error rather than correcting it by guesswork.

### Conclusion

The trained cross-encoder performs substantially better than the pretrained sentence-similarity system on held-out anchors. The controlled context changes are effectively tied at this resolution. Main weaknesses are confusing shared wording with equivalent meaning and missing technical relationships. More seeds would be needed to support a context advantage; no optional tuning runs or hidden-label lookups were used.

### Submission status

Tasks 1-4 use the measured local experiments. The validation-selected raw-CPC configuration also completed an offline Kaggle GPU run and a scored late submission on 12 November 2026. Local validation r = 0.8195; the Kaggle execution gives r = 0.8192. The final notebook is public, and the separate Submissions screenshot records both hidden-test scores.

### References

- [Official brief and starter](https://github.com/bk-anupam/generative_ai_shared/tree/master/module3/labs/patent_phrase_matching)
- [Official competition data and rules](https://www.kaggle.com/competitions/us-patent-phrase-to-phrase-matching)
- [DeBERTa-v3-small model card; MIT](https://huggingface.co/microsoft/deberta-v3-small)
- [MiniLM sentence embedding model card; Apache-2.0](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)

Prepared with Codex assistance; observations are based on the saved actual runs. No other student solution was adapted.


## Revision provenance

Prepared with Codex assistance from the official course notebook. The student should review and understand the implementation and analysis before submission. No other student solution was read or adapted.

Local runtime: NVIDIA RTX 2000 Ada Generation (16 GB), PyTorch 2.10.0+cu128, Transformers 4.57.1, PEFT 0.17.1. Required workloads and splits are retained. Outputs, when present, come from actual local execution; empty or explicitly pending sections are not claimed as completed. The student explicitly approved the final Kaggle and repository submission on 12 November 2026.


## Final scored Kaggle submission

Tasks 1-4 use the measured local experiments. The validation-selected raw-CPC configuration also completed an offline Kaggle GPU run and a scored late submission on 12 November 2026. Local validation r = 0.8195; the Kaggle execution gives r = 0.8192. The final notebook is public, and the separate Submissions screenshot records both hidden-test scores.

One scored competition submission: version 2, script ID 368715520, public 0.7904, private 0.8102. No test score influenced model selection. The first saved Kaggle execution failed before training because Transformers 5.20 removed warmup_ratio; it was not submitted. The successful version uses the equivalent 272 warmup steps (10% of 2,712), one T4, and unchanged model, split, batch size and three-epoch budget.

Public minus Kaggle validation is -0.0288; private minus validation is -0.0090; public minus private is -0.0198. Different anchors, CPC-section mixes and sample sizes can explain gaps. Selecting among three configurations on one validation split can also make its winner optimistic. These are plausible explanations, not verified causes.

Kaggle validation differs from local by -0.0003. The T4/PyTorch 2.11/Transformers 5.20.2 environment differs from the local RTX 2000 Ada/Transformers 4.57.1 setup, so exact numerical replication is not expected. All controlled comparisons in Tasks 1-4 use one local environment.

[Final Kaggle version](https://www.kaggle.com/code/vanshjoshi73/lab-7-patent-phrase-matching-vansh-joshi?scriptVersionId=368715520)
